In [69]:
import pandas as pd
from datetime import datetime, date

## Decision Tree Classification

In [70]:
df_HR = pd.read_csv("HRDataset_v14.csv")

In [71]:
df_ES = pd.read_csv("Employee Attrition.csv")

In [72]:
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

#### HR Dataset

In [73]:
print(df_HR.shape)
print(df_HR.info())

(311, 36)
<class 'pandas.DataFrame'>
RangeIndex: 311 entries, 0 to 310
Data columns (total 36 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   Employee_Name               311 non-null    str    
 1   EmpID                       311 non-null    int64  
 2   MarriedID                   311 non-null    int64  
 3   MaritalStatusID             311 non-null    int64  
 4   GenderID                    311 non-null    int64  
 5   EmpStatusID                 311 non-null    int64  
 6   DeptID                      311 non-null    int64  
 7   PerfScoreID                 311 non-null    int64  
 8   FromDiversityJobFairID      311 non-null    int64  
 9   Salary                      311 non-null    int64  
 10  Termd                       311 non-null    int64  
 11  PositionID                  311 non-null    int64  
 12  Position                    311 non-null    str    
 13  State                       311 non-

In [74]:
columns_to_remove = ["Employee_Name", "EmpID", "DateofTermination", "TermReason", "EmploymentStatus", "EmpStatusID"]
df_HR = df_HR.drop(columns=[c for c in columns_to_remove if c in df_HR.columns])

In [75]:
def age(born):
    for fmt in ("%d/%m/%y", "%m/%d/%y"):
        try:
            born = datetime.strptime(born, fmt).date()
            break
        except ValueError:
            continue
    else:
        return None

    today = date.today()
    return today.year - born.year - ((today.month, today.day) < (born.month, born.day))

In [76]:
if "DOB" in df_HR.columns:
    df_HR["Age"] = df_HR["DOB"].apply(age)
    df_HR = df_HR.drop(columns=["DOB"])

In [77]:
target_col_HR = "Termd"
y_HR = df_HR[target_col_HR]
X_HR = df_HR.drop(columns=[target_col_HR])

In [78]:
HR_encoders = {}
for col in X_HR.select_dtypes(include="object").columns:
    le = LabelEncoder()
    X_HR[col] = le.fit_transform(X_HR[col].astype(str))
    HR_encoders[col] = le

C:\Users\Marc\AppData\Local\Temp\ipykernel_14692\641722634.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for col in X_HR.select_dtypes(include="object").columns:


In [79]:
print(y_HR.value_counts(normalize=True))

Termd
0    0.665595
1    0.334405
Name: proportion, dtype: float64


In [80]:
X_train_HR, X_test_HR, y_train_HR, y_test_HR = train_test_split(X_HR, y_HR, test_size=0.2, random_state=42, stratify=y_HR)

In [81]:
clf_HR = DecisionTreeClassifier(max_depth=5, random_state=42)
clf_HR.fit(X_train_HR, y_train_HR)

,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.",'gini'
,"splitter splitter: {""best"", ""random""}, default=""best""The strategy used to choose the split at each node. Supportedstrategies are ""best"" to choose the best split and ""random"" to choosethe best random split.",'best'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",5
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: int, float or {""sqrt"", ""log2""}, default=NoneThe number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... note:: The search for a split does not stop until at least one valid partition of the node samples is found, even if it requires to effectively inspect more than ``max_features`` features.",None
,"random_state random_state: int, RandomState instance or None, default=NoneControls the randomness of the estimator. The features are alwaysrandomly permuted at each split, even if ``splitter`` is set to``""best""``. When ``max_features < n_features``, the algorithm willselect ``max_features`` at random at each split before finding the bestsplit among them. But the best found split may vary across differentruns, even if ``max_features=n_features``. That is the case, if theimprovement of the criterion is identical for several splits and onesplit has to be selected at random. To obtain a deterministic behaviourduring fitting, ``random_state`` has to be fixed to an integer.See :term:`Glossary ` for details.",42
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow a tree with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current no

In [82]:
y_pred_HR = clf_HR.predict(X_test_HR)

In [83]:
print("HR Dataset - Accuracy:", accuracy_score(y_test_HR, y_pred_HR))
print("\nHR Dataset - Classification report:\n", classification_report(y_test_HR, y_pred_HR))
print("\nHR Dataset - Confusion matrix:\n", confusion_matrix(y_test_HR, y_pred_HR))

HR Dataset - Accuracy: 0.8571428571428571

HR Dataset - Classification report:
               precision    recall  f1-score   support

           0       0.85      0.95      0.90        42
           1       0.88      0.67      0.76        21

    accuracy                           0.86        63
   macro avg       0.86      0.81      0.83        63
weighted avg       0.86      0.86      0.85        63


HR Dataset - Confusion matrix:
 [[40  2]
 [ 7 14]]


In [84]:
importances_HR = pd.Series(clf_HR.feature_importances_, index=X_HR.columns).sort_values(ascending=False)
print("\nHR Dataset - Feature importances:\n", importances_HR)


HR Dataset - Feature importances:
 LastPerformanceReview_Date    0.654829
ManagerName                   0.076822
DateofHire                    0.076198
ManagerID                     0.073817
Salary                        0.047287
Absences                      0.029563
EngagementSurvey              0.023047
RecruitmentSource             0.018437
PerfScoreID                   0.000000
MaritalStatusID               0.000000
GenderID                      0.000000
DeptID                        0.000000
MarriedID                     0.000000
MaritalDesc                   0.000000
Sex                           0.000000
Zip                           0.000000
State                         0.000000
Position                      0.000000
PositionID                    0.000000
FromDiversityJobFairID        0.000000
RaceDesc                      0.000000
Department                    0.000000
CitizenDesc                   0.000000
HispanicLatino                0.000000
PerformanceScore            

#### ES Dataset

In [85]:
print(df_ES.shape)
print(df_ES.info)
print(df_ES.columns.tolist)

(15787, 10)
<bound method DataFrame.info of         Emp ID  satisfaction_level  last_evaluation  number_project  \
0          1.0                0.38             0.53             2.0   
1          2.0                0.80             0.86             5.0   
2          3.0                0.11             0.88             7.0   
3          4.0                0.72             0.87             5.0   
4          5.0                0.37             0.52             2.0   
...        ...                 ...              ...             ...   
15782  14995.0                0.40             0.57             2.0   
15783  14996.0                0.37             0.48             2.0   
15784  14997.0                0.37             0.53             2.0   
15785  14998.0                0.11             0.96             6.0   
15786  14999.0                0.37             0.52             2.0   

       average_montly_hours  time_spend_company  Work_accident  \
0                     157.0          

In [86]:
df_ES = df_ES.dropna(how="all")
print(df_ES.shape)

(14999, 10)


In [87]:
satisfaction_col = "satisfaction_level"

In [88]:
df_ES["satisfaction_class"] = pd.cut(
    df_ES[satisfaction_col],
    bins=[-0.01, 0.4, 0.7, 1.0],
    labels=["Low", "Medium", "High"]
)

In [89]:
target_col_ES = "satisfaction_class"
y_ES = df_ES[target_col_ES]
X_ES = df_ES.drop(columns=[target_col_ES, "satisfaction_level", "Emp ID"])

In [90]:
ES_encoders = {}
for col in X_ES.select_dtypes(include="object").columns:
    le = LabelEncoder()
    X_ES[col] = le.fit_transform(X_ES[col].astype(str))
    ES_encoders[col] = le

C:\Users\Marc\AppData\Local\Temp\ipykernel_14692\3338801408.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for col in X_ES.select_dtypes(include="object").columns:


In [91]:
print(y_ES.value_counts(normalize=True))

satisfaction_class
High      0.419895
Medium    0.371825
Low       0.208281
Name: proportion, dtype: float64


In [92]:
X_train_ES, X_test_ES, y_train_ES, y_test_ES = train_test_split(X_ES, y_ES, test_size=0.2, random_state=42, stratify=y_ES)

In [93]:
clf_ES = DecisionTreeClassifier(max_depth=5, random_state=42)
clf_ES.fit(X_train_ES, y_train_ES)

,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.",'gini'
,"splitter splitter: {""best"", ""random""}, default=""best""The strategy used to choose the split at each node. Supportedstrategies are ""best"" to choose the best split and ""random"" to choosethe best random split.",'best'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",5
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: int, float or {""sqrt"", ""log2""}, default=NoneThe number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... note:: The search for a split does not stop until at least one valid partition of the node samples is found, even if it requires to effectively inspect more than ``max_features`` features.",None
,"random_state random_state: int, RandomState instance or None, default=NoneControls the randomness of the estimator. The features are alwaysrandomly permuted at each split, even if ``splitter`` is set to``""best""``. When ``max_features < n_features``, the algorithm willselect ``max_features`` at random at each split before finding the bestsplit among them. But the best found split may vary across differentruns, even if ``max_features=n_features``. That is the case, if theimprovement of the criterion is identical for several splits and onesplit has to be selected at random. To obtain a deterministic behaviourduring fitting, ``random_state`` has to be fixed to an integer.See :term:`Glossary ` for details.",42
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow a tree with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current no

In [94]:
y_pred_ES = clf_ES.predict(X_test_ES)

In [95]:
print("Satisfaction Dataset - Accuracy:", accuracy_score(y_test_ES, y_pred_ES))
print("\nSatisfaction Dataset - Classification report:\n", classification_report(y_test_ES, y_pred_ES))
print("\nSatisfaction Dataset - Confusion matrix:\n", confusion_matrix(y_test_ES, y_pred_ES))

Satisfaction Dataset - Accuracy: 0.557

Satisfaction Dataset - Classification report:
               precision    recall  f1-score   support

        High       0.55      0.89      0.68      1260
         Low       0.69      0.46      0.55       625
      Medium       0.50      0.23      0.32      1115

    accuracy                           0.56      3000
   macro avg       0.58      0.53      0.51      3000
weighted avg       0.56      0.56      0.52      3000


Satisfaction Dataset - Confusion matrix:
 [[1126   71   63]
 [ 138  286  201]
 [ 799   57  259]]


In [96]:
importances_ES = pd.Series(clf_ES.feature_importances_, index=X_ES.columns).sort_values(ascending=False)
print("\nSatisfaction Dataset - Feature importances:\n", importances_ES)


Satisfaction Dataset - Feature importances:
 number_project           0.627588
average_montly_hours     0.186865
last_evaluation          0.095863
time_spend_company       0.084498
dept                     0.003927
promotion_last_5years    0.001259
Work_accident            0.000000
salary                   0.000000
dtype: float64


## Support Vector Representation

In [97]:
from sklearn.svm import SVR
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

#### HR Dataset

In [98]:
target_col_HR_reg = "Salary"
y_HR_reg = df_HR[target_col_HR_reg]
X_HR_reg = df_HR.drop(columns=[target_col_HR_reg])

In [99]:
print(X_HR_reg.isnull().sum()[X_HR_reg.isnull().sum() > 0])

ManagerID    8
dtype: int64


In [100]:
num_cols_HR_reg = X_HR_reg.select_dtypes(include="number").columns

In [101]:
num_cols_HR_reg = X_HR_reg.select_dtypes(include="number").columns
X_HR_reg[num_cols_HR_reg] = X_HR_reg[num_cols_HR_reg].fillna(X_HR_reg[num_cols_HR_reg].median())

cat_cols_HR_reg = X_HR_reg.select_dtypes(include="object").columns
for col in cat_cols_HR_reg:
    X_HR_reg[col] = X_HR_reg[col].fillna(X_HR_reg[col].mode()[0])

C:\Users\Marc\AppData\Local\Temp\ipykernel_14692\1601450397.py:4: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols_HR_reg = X_HR_reg.select_dtypes(include="object").columns


In [102]:
HR_reg_encoders = {}
for col in X_HR_reg.select_dtypes(include="object").columns:
    le = LabelEncoder()
    X_HR_reg[col] = le.fit_transform(X_HR_reg[col].astype(str))
    HR_reg_encoders[col] = le

C:\Users\Marc\AppData\Local\Temp\ipykernel_14692\1357726967.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for col in X_HR_reg.select_dtypes(include="object").columns:


In [103]:
X_train_HR_reg, X_test_HR_reg, y_train_HR_reg, y_test_HR_reg = train_test_split(X_HR_reg, y_HR_reg, test_size=0.2, random_state=42)

In [104]:
scaler_HR = StandardScaler()
X_train_HR_scaled = scaler_HR.fit_transform(X_train_HR_reg)
X_test_HR_scaled = scaler_HR.transform(X_test_HR_reg)

In [105]:
svr_HR = SVR(kernel="rbf")
svr_HR.fit(X_train_HR_scaled, y_train_HR_reg)

,"kernel kernel: {'linear', 'poly', 'rbf', 'sigmoid', 'precomputed'} or callable, default='rbf'Specifies the kernel type to be used in the algorithm.If none is given, 'rbf' will be used. If a callable is given it isused to precompute the kernel matrix.For an intuitive visualization of different kernel typessee :ref:`sphx_glr_auto_examples_svm_plot_svm_regression.py`",'rbf'
,"degree degree: int, default=3Degree of the polynomial kernel function ('poly').Must be non-negative. Ignored by all other kernels.",3
,"gamma gamma: {'scale', 'auto'} or float, default='scale'Kernel coefficient for 'rbf', 'poly' and 'sigmoid'.- if ``gamma='scale'`` (default) is passed then it uses 1 / (n_features * X.var()) as value of gamma,- if 'auto', uses 1 / n_features- if float, must be non-negative... versionchanged:: 0.22 The default value of ``gamma`` changed from 'auto' to 'scale'.",'scale'
,"coef0 coef0: float, default=0.0Independent term in kernel function.It is only significant in 'poly' and 'sigmoid'.",0.0
,"tol tol: float, default=1e-3Tolerance for stopping criterion.",0.001
,"C C: float, default=1.0Regularization parameter. The strength of the regularization isinversely proportional to C. Must be strictly positive.The penalty is a squared l2. For an intuitive visualization of theeffects of scaling the regularization parameter C, see:ref:`sphx_glr_auto_examples_svm_plot_svm_scale_c.py`.",1.0
,"epsilon epsilon: float, default=0.1Epsilon in the epsilon-SVR model. It specifies the epsilon-tubewithin which no penalty is associated in the training loss functionwith points predicted within a distance epsilon from the actualvalue. Must be non-negative.",0.1
,"shrinking shrinking: bool, default=TrueWhether to use the shrinking heuristic.See the :ref:`User Guide `.",True
,"cache_size cache_size: float, default=200Specify the size of the kernel cache (in MB).",200
,"verbose verbose: bool, default=FalseEnable verbose output. Note that this setting takes advantage of aper-process runtime setting in libsvm that, if enabled, may not workproperly in a multithreaded context.",False
,"max_iter max_iter: int, default=-1Hard limit on iterations within solver, or -1 for no limit.",-1


In [106]:
y_pred_HR_reg = svr_HR.predict(X_test_HR_scaled)

In [107]:
print("HR Dataset (SVR) - R2:", r2_score(y_test_HR_reg, y_pred_HR_reg))
print("HR Dataset (SVR) - MSE:", mean_squared_error(y_test_HR_reg, y_pred_HR_reg))
print("HR Dataset (SVR) - MAE:", mean_absolute_error(y_test_HR_reg, y_pred_HR_reg))

HR Dataset (SVR) - R2: -0.10544698489789761
HR Dataset (SVR) - MSE: 894777247.702201
HR Dataset (SVR) - MAE: 15972.57608909898


#### ES Dataset

In [108]:
target_col_ES_reg = "satisfaction_level"
y_ES_reg = df_ES[target_col_ES_reg]
X_ES_reg = df_ES.drop(columns=[target_col_ES_reg, "satisfaction_class", "Emp ID"])

In [109]:
ES_reg_encoders = {}
for col in X_ES_reg.select_dtypes(include="object").columns:
    le = LabelEncoder()
    X_ES_reg[col] = le.fit_transform(X_ES_reg[col].astype(str))
    ES_reg_encoders[col] = le

C:\Users\Marc\AppData\Local\Temp\ipykernel_14692\3629821437.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for col in X_ES_reg.select_dtypes(include="object").columns:


In [110]:
X_train_ES_reg, X_test_ES_reg, y_train_ES_reg, y_test_ES_reg = train_test_split(X_ES_reg, y_ES_reg, test_size=0.2, random_state=42)

In [111]:
scaler_ES = StandardScaler()
X_train_ES_scaled = scaler_ES.fit_transform(X_train_ES_reg)
X_test_ES_scaled = scaler_ES.transform(X_test_ES_reg)

In [112]:
svr_ES = SVR(kernel="rbf")
svr_ES.fit(X_train_ES_scaled, y_train_ES_reg)

,"kernel kernel: {'linear', 'poly', 'rbf', 'sigmoid', 'precomputed'} or callable, default='rbf'Specifies the kernel type to be used in the algorithm.If none is given, 'rbf' will be used. If a callable is given it isused to precompute the kernel matrix.For an intuitive visualization of different kernel typessee :ref:`sphx_glr_auto_examples_svm_plot_svm_regression.py`",'rbf'
,"degree degree: int, default=3Degree of the polynomial kernel function ('poly').Must be non-negative. Ignored by all other kernels.",3
,"gamma gamma: {'scale', 'auto'} or float, default='scale'Kernel coefficient for 'rbf', 'poly' and 'sigmoid'.- if ``gamma='scale'`` (default) is passed then it uses 1 / (n_features * X.var()) as value of gamma,- if 'auto', uses 1 / n_features- if float, must be non-negative... versionchanged:: 0.22 The default value of ``gamma`` changed from 'auto' to 'scale'.",'scale'
,"coef0 coef0: float, default=0.0Independent term in kernel function.It is only significant in 'poly' and 'sigmoid'.",0.0
,"tol tol: float, default=1e-3Tolerance for stopping criterion.",0.001
,"C C: float, default=1.0Regularization parameter. The strength of the regularization isinversely proportional to C. Must be strictly positive.The penalty is a squared l2. For an intuitive visualization of theeffects of scaling the regularization parameter C, see:ref:`sphx_glr_auto_examples_svm_plot_svm_scale_c.py`.",1.0
,"epsilon epsilon: float, default=0.1Epsilon in the epsilon-SVR model. It specifies the epsilon-tubewithin which no penalty is associated in the training loss functionwith points predicted within a distance epsilon from the actualvalue. Must be non-negative.",0.1
,"shrinking shrinking: bool, default=TrueWhether to use the shrinking heuristic.See the :ref:`User Guide `.",True
,"cache_size cache_size: float, default=200Specify the size of the kernel cache (in MB).",200
,"verbose verbose: bool, default=FalseEnable verbose output. Note that this setting takes advantage of aper-process runtime setting in libsvm that, if enabled, may not workproperly in a multithreaded context.",False
,"max_iter max_iter: int, default=-1Hard limit on iterations within solver, or -1 for no limit.",-1


In [113]:
y_pred_ES_reg = svr_ES.predict(X_test_ES_scaled)

In [114]:
print("ES Dataset (SVR) - R2:", r2_score(y_test_ES_reg, y_pred_ES_reg))
print("ES Dataset (SVR) - MSE:", mean_squared_error(y_test_ES_reg, y_pred_ES_reg))
print("ES Dataset (SVR) - MAE:", mean_absolute_error(y_test_ES_reg, y_pred_ES_reg))

ES Dataset (SVR) - R2: 0.3922702985580564
ES Dataset (SVR) - MSE: 0.03713473916903776
ES Dataset (SVR) - MAE: 0.14680976224814415
